# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a step-by-step guide for loading and exploring the FAIR^2 protein dataset using the [`mlcroissant`](https://mlcommons.github.io/croissant/) library.

### Dataset Source

The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant --quiet

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print the dataset name and description
print(f"Dataset: {dataset.metadata.name}\n")
print(f"Description: {dataset.metadata.description}\n")

## 2. Data Overview

Review available record sets, fields, and their IDs using the Croissant schema.

In [ ]:
# List all record sets by their @id
print("Available Record Sets (by @id):\n")
for record_set in dataset.record_sets:
    print(f"- @id: {record_set.id}, name: {record_set.name}")

# For demonstration, show the fields and their @id for each record set
print("\nFields for each Record Set:")
for record_set in dataset.record_sets:
    print(f"\nRecord Set: {record_set.name} (@id: {record_set.id})")
    for field in record_set.fields:
        print(f"    Field: {field.name}, @id: {field.id}, Data Type: {field.data_type}")

### Preview Records from One Record Set

Let's print a few records from a representative record set using its `@id`.

In [ ]:
# Pick the main record set for proteins. Find the @id programmatically or set manually:

# Find a record set containing 'protein' or show all for demonstration:
selected_record_set_id = None
for record_set in dataset.record_sets:
    if 'protein' in record_set.name.lower() or 'abundance' in record_set.name.lower():
        selected_record_set_id = record_set.id
        break
if selected_record_set_id is None:
    # Fallback: use the first record set
    selected_record_set_id = dataset.record_sets[0].id
print(f"Selected record set for preview: {selected_record_set_id}\n")

# Show a brief preview of the first 2 records
for i, record in enumerate(dataset.records(record_set=selected_record_set_id)):
    print(record)
    if i == 1:
        break

## 3. Data Extraction

Load data from all available record sets into separate DataFrames using their `@id` fields.

In [ ]:
record_sets_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}

for rs_id in record_sets_ids:
    print(f"Loading records for Record Set {rs_id} ...")
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df

print(f"Loaded DataFrames: {list(dataframes.keys())}\n")

# Choose the same record set as before
main_rs_id = selected_record_set_id
print(f"Columns in main record set ({main_rs_id}):")
print(dataframes[main_rs_id].columns.tolist())

# Display top 5 rows of main DataFrame
dataframes[main_rs_id].head()

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Below, we select a numeric field and a group field from the main record set for demonstration.

In [ ]:
df = dataframes[main_rs_id]

# Try to automatically select a numeric field (float or int)
numeric_field = None
for col in df.columns:
    try:
        # Try converting to numeric to check suitability
        sample = pd.to_numeric(df[col], errors='coerce')
        if sample.notnull().sum() > 0:
            numeric_field = col
            break
    except:
        continue

if numeric_field is None:
    print("No numeric field found for EDA.")
else:
    print(f"Selected numeric field: {numeric_field}")
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
    threshold = df[numeric_field].mean()  # Use mean as threshold
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Number of records with {numeric_field} > {threshold:.2f}: {len(filtered_df)}")
    display(filtered_df.head())

    # Normalize the numeric field for filtered records
    filtered_df[f"{numeric_field}_normalized"] = (
        (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) /
        filtered_df[numeric_field].std()
    )
    print(f"\nNormalized {numeric_field} (first 5 rows):")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try to select a textual/categorical field for grouping
    group_field = None
    for col in df.columns:
        if df[col].dtype == object and col != numeric_field:
            group_field = col
            break

    if group_field is not None:
        print(f"\nGrouping by: {group_field}")
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        display(grouped_df.head())

## 5. Visualization

Visualize data distributions or relationships between fields in the dataset. The examples below use Matplotlib and Seaborn.

> **Note:** You may need to adjust the chosen column names based on field availability.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot the distribution of the selected numeric field
if numeric_field is not None and numeric_field in df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.show()

    # If grouping field is set, plot group means
    if group_field is not None and group_field in df.columns:
        mean_by_group = df.groupby(group_field)[numeric_field].mean().sort_values()
        mean_by_group.plot(kind='barh', figsize=(8, 6))
        plt.title(f'Average {numeric_field} by {group_field}')
        plt.xlabel(f'Average {numeric_field}')
        plt.ylabel(group_field)
        plt.show()

## 6. Conclusion

In this notebook, we loaded the FAIR<sup>2</sup> protein dataset using `mlcroissant`, explored its structure via record sets and fields, and performed basic data cleaning, normalization, grouping, and visualization of protein measurement values. For more advanced analysis, you can select specific fields by their `@id` and apply domain-specific statistics or machine learning models.\

Remember to always use field and record set `@id`s for reference and extraction when working with Croissant-based datasets.